In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer,SimpleImputer     
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,StandardScaler,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report ,accuracy_score
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV      
from sklearn.pipeline import Pipeline,make_pipeline     
from sklearn.svm import SVC      
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier           
import matplotlib.pyplot as plt
import seaborn as sns     

In [ ]:
df=pd.read_csv("/kaggle/input/titanic/train.csv",index_col="PassengerId")
df.head()

In [ ]:
df.info()

In [ ]:
numric_head=df.select_dtypes(np.number)
numric_head.shape

In [ ]:
numric_head.shape[1]

In [ ]:
fig ,ax=plt.subplots(nrows=3,ncols=3,figsize=(9,9))
x=0
for i in range(0,3):
    for j in range(0,3):
        if x>=numric_head.shape[1]:
            ax[i, j].set_visible(False)
        else:
            ax[i,j].hist(numric_head.iloc[:,x]);
            ax[i,j].set(xlabel=numric_head.columns[x])
            x+=1

In [ ]:
df.drop("Cabin",axis=1,inplace=True)
df.info()

In [ ]:
df["Pclass"]=df["Pclass"].astype("object")
df.info()

In [ ]:
X=df.iloc[:,1:]
Y=df.iloc[:,0]

x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=14,shuffle=True)
len(x_train)


In [ ]:
n_column=x_train.select_dtypes(np.number)
n_column_name=n_column.columns.tolist()
un_column=x_train.select_dtypes(exclude=(np.number))
un_column_name=un_column.columns.tolist()
or_un_column=pd.DataFrame(un_column["Pclass"])
un_column.drop("Pclass",axis=1,inplace=True)
or_un_column_name=or_un_column.columns.to_list()
or_un_column_name

In [ ]:
st=StandardScaler()
kn=KNNImputer(n_neighbors=5)
sip=SimpleImputer(strategy="most_frequent")

In [ ]:
numeric_pipeline=Pipeline(steps=[
    ("numeric_imputer",StandardScaler()),
    ("encoder",KNNImputer(n_neighbors=5))

])

categorical_pipeline=Pipeline(steps=[
    ("categorical_imputer",SimpleImputer(strategy="most_frequent")),
    ("LabelEncod",OneHotEncoder(handle_unknown="ignore"))
])



ordinal_column_pipline=Pipeline(steps=[
    ("OrdinalEncod",OrdinalEncoder(categories=[[1,2,3]])),
    ("ordinal_column_imputer",StandardScaler())
])

In [ ]:
preprocessor=ColumnTransformer(
    transformers=[
        ("num_1",numeric_pipeline,n_column_name),
        ("categorical",categorical_pipeline,un_column_name),
        ("Ordinal",ordinal_column_pipline,or_un_column_name)
        
    ]
)

In [ ]:
# df=pd.DataFrame(preprocessor.fit_transform(df),columns=df.columns,index=df.index)
# df.info()

In [ ]:
x_train=preprocessor.fit_transform(x_train)

In [ ]:
model_selected={
    "SVC":{
        "model": SVC(random_state=43),
        "hp":{
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf'],
            'gamma': ['scale', 'auto']
        }
    },
    "RandomForestClassifier":{
        "model": RandomForestClassifier(random_state=43),
        "hp":{
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10]
        }
    },
    "DecisionTreeClassifier":{
        "model": DecisionTreeClassifier(random_state=43),
        "hp":{
            "max_depth":[None,2,5,10],
            "criterion":["entropy","gini"],
            "splitter":["random","best"]
        }
    }
    ,"KNeighborsClassifier": {
        'model': KNeighborsClassifier(),
        'hp': {
            'n_neighbors': [3, 5, 7, 9],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    }

        
    
}

In [ ]:
results=[]
for model_name,mhp in model_selected.items():
    print(f"شروع جستجو برای مدل: {model_name}...")
    grid=GridSearchCV(
        estimator=mhp["model"],
        param_grid=mhp["hp"],
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
        verbose=1,
        )
    grid.fit(x_train,y_train)
    results.append({
        'model': model_name,
        'best_score_cv': grid.best_score_,
        'best_params': grid.best_params_,
        'best_estimator': grid.best_estimator_})
print("\nجستجو برای تمام مدل‌ها به پایان رسید.")

In [ ]:
x_test_scaled=preprocessor.transform(x_test)
len(x_test)

In [ ]:
# تبدیل نتایج به یک DataFrame برای تحلیل بهتر
results_df = pd.DataFrame(results, columns=['model', 'best_score_cv', 'best_params'])

# مرتب‌سازی نتایج بر اساس بهترین امتیاز (نزولی)
results_df = results_df.sort_values(by='best_score_cv', ascending=False).reset_index(drop=True)

print("\n--- نتایج کلی پایپ‌لاین ---")
print(results_df)

# شناسایی بهترین مدل کلی
best_model_name = results_df.loc[0, 'model']
best_model_score = results_df.loc[0, 'best_score_cv']
best_model_params = results_df.loc[0, 'best_params']

print(f"\n✅ بهترین مدل پیدا شده: {best_model_name}")
print(f"✅ بهترین امتیاز دقت (با Cross-Validation): {best_model_score:.4f}")
print(f"✅ بهترین هایپرپارامترها: {best_model_params}")

# پیدا کردن بهترین تخمین‌گر (estimator) از لیست نتایج
best_estimator = None
for res in results:
    if res['model'] == best_model_name:
        best_estimator = res['best_estimator']
        break

# ارزیابی نهایی بهترین مدل روی داده‌های آزمون (که تا به حال دیده نشده)
y_pred = best_estimator.predict(x_test_scaled)
final_accuracy = accuracy_score(y_test, y_pred)

print(f"\n🚀 دقت نهایی بهترین مدل روی داده‌های آزمون: {final_accuracy:.4f}")